In [17]:
from pathlib import Path
import pandas as pd

csv_path = Path("../clean_data/parp_cleand_new.csv")
if not csv_path.exists():
    raise FileNotFoundError(csv_path)

df = pd.read_csv(csv_path)

if "TEST MON TH" in df.columns and "TEST MONTH" not in df.columns:
    df = df.rename(columns={"TEST MON TH": "TEST MONTH"})

if "TEST YR" in df.columns and "TEST YEAR" not in df.columns:
    df = df.rename(columns={"TEST YR": "TEST YEAR"})

if "TEST DY" in df.columns and "TEST DAY" not in df.columns:
    df = df.rename(columns={"TEST DY": "TEST DAY"})


def clean_date_part(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    try:
        num = float(text)
    except ValueError:
        return text
    if num.is_integer():
        return str(int(num))
    return text


required = ["TEST DAY", "TEST MONTH", "TEST YEAR"]
if not all(c in df.columns for c in required):
    print("Missing date columns. Found:", df.columns.tolist())
else:
    day = df["TEST DAY"].apply(clean_date_part)
    month = df["TEST MONTH"].apply(clean_date_part)
    year = df["TEST YEAR"].apply(clean_date_part)
    date_text = day + "/" + month + "/" + year
    df["TEST DATE"] = pd.to_datetime(date_text, errors="coerce", dayfirst=True).dt.date

for col in df.columns:
    if col == "TEST DATE":
        continue
    df[col] = df[col].replace(r"^\s*$", 0, regex=True).fillna(0)

df = df.drop(columns=[c for c in ["Column1", "Column26"] if c in df.columns], errors="ignore")

df.to_csv(csv_path, index=False)
print(f"Wrote {len(df)} rows to {csv_path}")

df.head()

Wrote 1083 rows to ..\clean_data\parp_cleand_new.csv


,source,sheet,CYCLE,FIELD NAME,RESVR,RESERVOIR DETAILED NAME,WELL,TEST DAY,TEST MONTH,TEST YEAR,...,DAILY OIL BBLS,DAILY WATER BBLS,DAILY NETGAS SMCF,DAILY GLGAS SMCF,PROD DAYS,PROD HRS,PCT WTR,NET GOR,WELL REMARK,TEST DATE
0,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,12,1,1995,...,880,0,0,0,28,0,0.0,0,PREV TEST,1995-01-12
1,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BASS,BAHI SAND STONE,A-001-32,0,0,0,...,0,0,0,0,0,0,0.0,0,SHUT-IN,NaT
2,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BASS,BAHI SAND STONE,A-069-32,0,0,0,...,0,0,0,0,0,0,0.0,0,NOW SUSPEND,NaT
3,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,15,2,1996,...,724,222,0,0,22,0,23.5,0,NEW WELL,1996-02-15
4,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BPL7,PALEOCENE,A-002-32,0,0,0,...,0,0,0,0,0,0,0.0,0,ABANDONED,NaT


# OAPARP25 cleaning notebook
Edit the config in the first code cell if needed, then run it.

In [18]:
from pathlib import Path
import pandas as pd

# Config
INPUT_ROOT = Path("../PARP")
OUTPUT_CSV = Path("../clean_data/parp_cleand_new.csv")
SHEET_NAME = "OAPARP25"

GROUP_HEADERS = {
    "WELL TEST DATE",
    "WELL STATUS",
    "WELL TEST DATA",
    "WELL TEST",
}
SUBHEADER_TOKENS = {
    "MTH",
    "DAY",
    "YR",
    "HRS",
    "CPR",
    "TPR",
    "SEP",
    "BOPD",
    "BWPD",
    "GOR",
}
HEADER_ALIASES = {
    "TEST MON TH": "TEST MONTH",
    "TEST MON": "TEST MONTH",
    "TEST YR": "TEST YEAR",
    "TEST DY": "TEST DAY",
}


def normalize_col_name(value):
    text = "" if pd.isna(value) else str(value)
    text = " ".join(text.replace("\n", " ").split())
    return text.strip()


def build_headers(row):
    headers = []
    for idx, value in enumerate(row):
        name = "" if pd.isna(value) else str(value).strip()
        if not name or name.lower() == "nan":
            name = f"Column{idx + 1}"
        headers.append(name)
    return headers


def make_unique_columns(columns):
    counts = {}
    unique_cols = []
    for col in columns:
        base = str(col)
        if base in counts:
            counts[base] += 1
            unique_cols.append(f"{base}.{counts[base]}")
        else:
            counts[base] = 0
            unique_cols.append(base)
    return unique_cols


def is_subheader_row(row):
    values = {
        normalize_col_name(v).upper()
        for v in row.tolist()
        if pd.notna(v) and normalize_col_name(v)
    }
    return bool(values & SUBHEADER_TOKENS)


def merge_headers(primary, secondary):
    merged = list(primary)
    for idx, sub in enumerate(secondary):
        sub_clean = normalize_col_name(sub)
        if not sub_clean or sub_clean.lower().startswith("column"):
            continue
        prim_clean = normalize_col_name(primary[idx]).upper()
        if not prim_clean or prim_clean.startswith("COLUMN") or prim_clean in GROUP_HEADERS:
            merged[idx] = sub_clean
    return merged


def locate_header_index(df):
    for idx, row in df.iterrows():
        values = {
            normalize_col_name(v).upper()
            for v in row.tolist()
            if pd.notna(v) and normalize_col_name(v)
        }
        has_field = "FIELD" in values or "FIELD NAME" in values
        has_resvr = "RESVR" in values or "RESVR CODE" in values
        has_date = (
            "TEST YEAR" in values
            or "YR" in values
            or "TEST MONTH" in values
            or "TEST DAY" in values
            or "WELL TEST DAY" in values
        )
        if has_field and has_resvr and has_date:
            return idx
    return None


def normalize_year_value(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text:
        return pd.NA
    try:
        number = int(float(text))
    except ValueError:
        return value
    if number == 0:
        return 2000
    if number == 1:
        return 2001
    if number > 60:
        return 1900 + number
    if number < 27:
        return 2000 + number
    return number


def clean_date_part(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    try:
        num = float(text)
    except ValueError:
        return text
    if num.is_integer():
        return str(int(num))
    return text


def parse_sheet(df):
    header_idx = locate_header_index(df)
    if header_idx is None:
        return pd.DataFrame()

    start = df.loc[header_idx:].copy()
    header_row = start.iloc[0]
    subheader_row = start.iloc[1] if len(start) > 1 else None

    headers = build_headers(header_row)
    if subheader_row is not None and is_subheader_row(subheader_row):
        subheaders = build_headers(subheader_row)
        headers = merge_headers(headers, subheaders)
        data = start.iloc[2:].copy()
    else:
        data = start.iloc[1:].copy()

    headers = [normalize_col_name(h) for h in headers]
    headers = [HEADER_ALIASES.get(h, h) for h in headers]
    data.columns = make_unique_columns(headers)
    data = data.dropna(how="all")
    return data


def clean_sheet(path):
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(path, engine=engine) as xl:
        if SHEET_NAME not in xl.sheet_names:
            return pd.DataFrame()
        df = pd.read_excel(xl, sheet_name=SHEET_NAME, header=None, dtype=str)

    data = parse_sheet(df)
    if data.empty:
        return data

    if "WELL" in data.columns:
        well_text = data["WELL"].astype(str).str.strip()
        data = data[data["WELL"].notna() & (well_text != "") & (well_text.str.lower() != "nan")]

    for col in ["TEST YEAR", "YR", "YEAR"]:
        if col in data.columns:
            data[col] = data[col].apply(normalize_year_value)

    date_parts = ["TEST DAY", "TEST MONTH", "TEST YEAR"]
    if all(part in data.columns for part in date_parts):
        day = data["TEST DAY"].apply(clean_date_part)
        month = data["TEST MONTH"].apply(clean_date_part)
        year = data["TEST YEAR"].apply(clean_date_part)
        date_text = day + "/" + month + "/" + year
        data["TEST DATE"] = pd.to_datetime(date_text, errors="coerce", dayfirst=True).dt.date

    for col in data.columns:
        if col == "TEST DATE":
            continue
        data[col] = data[col].replace(r"^\s*$", 0, regex=True).fillna(0)

    rename_map = {
        "WELL.1": "WELL TYPE",
        "GTY &": "GRAVITY",
        "BOPD": "DAILY OIL PRODUCTION",
        "BWPD": "WATER PRODUCTION",
        "PERC": "BSW",
    }
    data = data.rename(columns={k: v for k, v in rename_map.items() if k in data.columns})

    drop_cols = ["...", "WELL TEST DATA", "Column19", "Column1", "Column26"]
    data = data.drop(columns=[c for c in drop_cols if c in data.columns], errors="ignore")

    for col in data.select_dtypes(include="object").columns:
        data[col] = data[col].astype(str).str.strip()

    data.insert(0, "sheet", SHEET_NAME)
    data.insert(0, "source", path.name)

    return data

In [19]:
files = sorted([*INPUT_ROOT.rglob("*.xls"), *INPUT_ROOT.rglob("*.xlsx")])
print("files", len(files))

path = None
for p in files:
    engine = "xlrd" if p.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(p, engine=engine) as xl:
        if SHEET_NAME in xl.sheet_names:
            path = p
            break

print("using", path)

if path is not None:
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(path, engine=engine) as xl:
        df_raw = pd.read_excel(xl, sheet_name=SHEET_NAME, header=None, dtype=str)
    df_raw.head(12)

files 219
using ..\PARP\1996\02 FEB 1996 PARP REPORTS.xls


In [20]:
header_idx = locate_header_index(df_raw) if "df_raw" in globals() else None
header_idx

if header_idx is not None:
    display(df_raw.loc[header_idx])
    if header_idx + 1 < len(df_raw):
        display(df_raw.loc[header_idx + 1])

0                       CYCLE
1                  FIELD NAME
2                       RESVR
3     RESERVOIR DETAILED NAME
4                        WELL
5                    TEST DAY
6                  TEST MONTH
7                   TEST YEAR
8        TEST LENGTH IN HOURS
9                         MTD
10                        EXT
11                  PRES HEAD
12                  PSIG SEP`
13                CHOKES PROD
14                   CHOKE GL
15                   GRAV API
16             DAILY OIL BBLS
17           DAILY WATER BBLS
18          DAILY NETGAS SMCF
19           DAILY GLGAS SMCF
20                  PROD DAYS
21                   PROD HRS
22                    PCT WTR
23                    NET GOR
24                WELL REMARK
25                        NaN
Name: 4, dtype: object

0     FEBRUARY -1996
1                NaN
2                NaN
3                NaN
4                NaN
5                NaN
6                NaN
7                NaN
8                NaN
9                NaN
10               NaN
11               NaN
12               NaN
13               NaN
14               NaN
15               NaN
16               NaN
17               NaN
18               NaN
19               NaN
20               NaN
21               NaN
22               NaN
23               NaN
24               NaN
25               NaN
Name: 5, dtype: object

In [21]:
data = parse_sheet(df_raw) if "df_raw" in globals() else pd.DataFrame()
print("columns", list(data.columns))
data.head()

columns ['CYCLE', 'FIELD NAME', 'RESVR', 'RESERVOIR DETAILED NAME', 'WELL', 'TEST DAY', 'TEST MONTH', 'TEST YEAR', 'TEST LENGTH IN HOURS', 'MTD', 'EXT', 'PRES HEAD', 'PSIG SEP`', 'CHOKES PROD', 'CHOKE GL', 'GRAV API', 'DAILY OIL BBLS', 'DAILY WATER BBLS', 'DAILY NETGAS SMCF', 'DAILY GLGAS SMCF', 'PROD DAYS', 'PROD HRS', 'PCT WTR', 'NET GOR', 'WELL REMARK', 'Column26']


,CYCLE,FIELD NAME,RESVR,RESERVOIR DETAILED NAME,WELL,TEST DAY,TEST MONTH,TEST YEAR,TEST LENGTH IN HOURS,MTD,...,DAILY OIL BBLS,DAILY WATER BBLS,DAILY NETGAS SMCF,DAILY GLGAS SMCF,PROD DAYS,PROD HRS,PCT WTR,NET GOR,WELL REMARK,Column26
5,FEBRUARY -1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1996-02,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,12,1,95,6,I,...,880,0,,,28,0,0,,PREV TEST,
7,1996-02,BAHI,BASS,BAHI SAND STONE,A-001-32,,,,,,...,,,,,0,0,,,SHUT-IN,
8,1996-02,BAHI,BASS,BAHI SAND STONE,A-069-32,,,,,,...,,,,,0,0,,,NOW SUSPEND,
9,1996-02,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,15,2,96,24,F,...,724,222,,,22,0,23.5,,NEW WELL,


In [22]:
work = data.copy()

if "TEST MON TH" in work.columns and "TEST MONTH" not in work.columns:
    work = work.rename(columns={"TEST MON TH": "TEST MONTH"})

if "TEST YR" in work.columns and "TEST YEAR" not in work.columns:
    work = work.rename(columns={"TEST YR": "TEST YEAR"})

if "TEST DY" in work.columns and "TEST DAY" not in work.columns:
    work = work.rename(columns={"TEST DY": "TEST DAY"})

if "WELL" in work.columns:
    well_text = work["WELL"].astype(str).str.strip()
    work = work[work["WELL"].notna() & (well_text != "") & (well_text.str.lower() != "nan")]

for col in ["TEST YEAR", "YR", "YEAR"]:
    if col in work.columns:
        work[col] = work[col].apply(normalize_year_value)

if all(col in work.columns for col in ["TEST DAY", "TEST MONTH", "TEST YEAR"]):
    day = work["TEST DAY"].apply(clean_date_part)
    month = work["TEST MONTH"].apply(clean_date_part)
    year = work["TEST YEAR"].apply(clean_date_part)
    date_text = day + "/" + month + "/" + year
    work["TEST DATE"] = pd.to_datetime(date_text, errors="coerce", dayfirst=True).dt.date
    work[["TEST DAY", "TEST MONTH", "TEST YEAR", "TEST DATE"]].head()
else:
    print("Missing date columns:", [c for c in ["TEST DAY", "TEST MONTH", "TEST YEAR"] if c not in work.columns])

for col in work.columns:
    if col == "TEST DATE":
        continue
    work[col] = work[col].replace(r"^\s*$", 0, regex=True).fillna(0)

C:\Users\OPS010\AppData\Local\Temp\ipykernel_31568\3666716002.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  work[col] = work[col].replace(r"^\s*$", 0, regex=True).fillna(0)
C:\Users\OPS010\AppData\Local\Temp\ipykernel_31568\3666716002.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  work[col] = work[col].replace(r"^\s*$", 0, regex=True).fillna(0)


In [23]:
rename_map = {
    "WELL.1": "WELL TYPE",
    "GTY &": "GRAVITY",
    "BOPD": "DAILY OIL PRODUCTION",
    "BWPD": "WATER PRODUCTION",
    "PERC": "BSW",
}
work = work.rename(columns={k: v for k, v in rename_map.items() if k in work.columns})

work = work.drop(
    columns=[c for c in ["...", "WELL TEST DATA", "Column19", "Column1", "Column26"] if c in work.columns],
    errors="ignore",
)

for col in work.select_dtypes(include="object").columns:
    work[col] = work[col].astype(str).str.strip()

work.insert(0, "sheet", SHEET_NAME)
work.insert(0, "source", path.name if "path" in globals() and path is not None else "")

work.head()

,source,sheet,CYCLE,FIELD NAME,RESVR,RESERVOIR DETAILED NAME,WELL,TEST DAY,TEST MONTH,TEST YEAR,...,DAILY OIL BBLS,DAILY WATER BBLS,DAILY NETGAS SMCF,DAILY GLGAS SMCF,PROD DAYS,PROD HRS,PCT WTR,NET GOR,WELL REMARK,TEST DATE
6,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,12,1,1995,...,880,0,0,0,28,0,0,0,PREV TEST,1995-01-12
7,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BASS,BAHI SAND STONE,A-001-32,0,0,0,...,0,0,0,0,0,0,0,0,SHUT-IN,NaT
8,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BASS,BAHI SAND STONE,A-069-32,0,0,0,...,0,0,0,0,0,0,0,0,NOW SUSPEND,NaT
9,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,15,2,1996,...,724,222,0,0,22,0,23.5,0,NEW WELL,1996-02-15
10,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BPL7,PALEOCENE,A-002-32,0,0,0,...,0,0,0,0,0,0,0,0,ABANDONED,NaT


In [24]:
output_path = OUTPUT_CSV
output_path.parent.mkdir(parents=True, exist_ok=True)
work.to_csv(output_path, index=False)
print(f"Wrote {len(work)} rows to {output_path}")
work.head()

Wrote 1083 rows to ..\clean_data\parp_cleand_new.csv


,source,sheet,CYCLE,FIELD NAME,RESVR,RESERVOIR DETAILED NAME,WELL,TEST DAY,TEST MONTH,TEST YEAR,...,DAILY OIL BBLS,DAILY WATER BBLS,DAILY NETGAS SMCF,DAILY GLGAS SMCF,PROD DAYS,PROD HRS,PCT WTR,NET GOR,WELL REMARK,TEST DATE
6,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,12,1,1995,...,880,0,0,0,28,0,0,0,PREV TEST,1995-01-12
7,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BASS,BAHI SAND STONE,A-001-32,0,0,0,...,0,0,0,0,0,0,0,0,SHUT-IN,NaT
8,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BASS,BAHI SAND STONE,A-069-32,0,0,0,...,0,0,0,0,0,0,0,0,NOW SUSPEND,NaT
9,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,15,2,1996,...,724,222,0,0,22,0,23.5,0,NEW WELL,1996-02-15
10,02 FEB 1996 PARP REPORTS.xls,OAPARP25,1996-02,BAHI,BPL7,PALEOCENE,A-002-32,0,0,0,...,0,0,0,0,0,0,0,0,ABANDONED,NaT
